<p><font size="6" color='grey'> <b>
KI-Agenten. Verstehen. Anwenden. Gestalten.
</b></font> </br></p>



<p><font size="5" color='grey'> <b>
Structured Output
</b></font> </br></p>

---

In [1]:
#@title 🔧 Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

# LangSmith Env-Vars VOR allen LangChain-Imports setzen
import os
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "M06-Structured-Output"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    load_prompt,
    show_trace
)
setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

✓ OPENAI_API_KEY erfolgreich gesetzt
✓ LANGSMITH_API_KEY erfolgreich gesetzt

Python Version: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]

Installierte LangChain- und LangGraph-Bibliotheken:
langchain                                1.2.10
langchain-chroma                         1.1.0
langchain-classic                        1.0.1
langchain-community                      0.4.1
langchain-core                           1.2.15
langchain-ollama                         1.0.1
langchain-openai                         1.1.10
langchain-text-splitters                 1.1.1
langgraph                                1.0.9
langgraph-checkpoint                     4.0.0
langgraph-prebuilt                       1.0.8
langgraph-sdk                            0.3.9

IP-Adresse: 34.87.153.67
Hostname: 67.153.87.34.bc.googleusercontent.com
Stadt: Singapore
Region: Singapore
Land: SG
Koordinaten: 1.2897,103.8501
Provider: AS396982 Google LLC
Postleitzahl: 018989
Zeitzone: Asia/Singapore


# 1 | Übersicht
---



In diesem Modul lernen Sie, wie Sie LLMs dazu bringen, **strukturierte Daten** statt freiem Text zurückzuliefern.

| Problem | Lösung |
|---|---|
| LLM gibt Freitext → schwer weiterverarbeitbar | `with_structured_output()` + Pydantic-Schema |
| Regex-Parsing → fehleranfällig | Automatische Validierung durch Pydantic |
| Kein Type-Checking | Typsichere Python-Objekte |


In [2]:
#@title
#@markdown   <p><font size="4" color='green'>  Prozessdiagramm</font> </br></p>

diagram = '''
flowchart LR
    INPUT["Freier Text\n'Emma ist 34, Ärztin'"]
    SCHEMA["Pydantic Schema\nPerson(name, alter, beruf)"]
    LLM["LLM\ngpt-4o-mini"]
    OUTPUT["Python-Objekt\nPerson(name=Emma, alter=34, ...)"]

    INPUT --> LLM
    SCHEMA --> LLM
    LLM --> OUTPUT
'''

mermaid(diagram, width=750)

# 2 | Pydantic Basics
---



Pydantic ist die Grundlage für strukturierte Ausgaben. Ein **Pydantic-Modell** definiert:

- **Felder** mit Typ-Annotationen (`str`, `int`, `List[str]`, …)
- **Beschreibungen** per `Field(description=...)` – das LLM liest diese!
- **Optionale Felder** mit `Optional[...]` und `default=None`
- **Validierungslogik** automatisch durch Pydantic

> 💡 **Merkregel:** Der `description`-Text in `Field()` ist der Prompt-Hinweis für das LLM. Je klarer die Beschreibung, desto besser das Ergebnis.

In [ ]:
import json
from typing import Optional, List, Literal
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model

# ── Konfigurationskonstanten ───────────────────────────────────────────────
MAX_RETRIES = 3   # API-Retry-Versuche bei transienten Fehlern (with_retry)

llm = init_chat_model("openai:gpt-4o-mini", temperature=0.0)

# Einfaches Schema
class Person(BaseModel):
    """Informationen über eine Person."""
    name: str = Field(description="Vollständiger Name der Person")
    alter: int = Field(description="Alter der Person in Jahren")
    beruf: Optional[str] = Field(default=None, description="Aktueller Beruf, falls bekannt")

# Schema inspizieren – was das LLM sieht
mprint("## Pydantic-Schema")
mprint(f"**Felder:** `{list(Person.model_fields.keys())}`")
schema_json = json.dumps(Person.model_json_schema(), indent=2, ensure_ascii=False)
mprint(f"**JSON Schema:**\n```json\n{schema_json}\n```")
print(f"   MAX_RETRIES = {MAX_RETRIES}")

# 3 | with_structured_output()
---



`with_structured_output()` bindet ein Pydantic-Schema direkt ans LLM:

```python
structured_llm = llm.with_structured_output(MeinSchema)
ergebnis = structured_llm.invoke("...")
# ergebnis ist ein MeinSchema-Objekt – typsicher!
```

**Was intern passiert:**
1. LangChain schickt das JSON Schema des Pydantic-Modells an das LLM
2. Das LLM füllt die Felder anhand der `description`-Texte
3. LangChain validiert die Antwort und gibt ein typsicheres Objekt zurück

In [ ]:
class Person(BaseModel):
    """Informationen ueber eine Person."""
    name: str = Field(description="Vollstaendiger Name der Person")
    alter: int = Field(description="Alter der Person in Jahren")
    beruf: str = Field(description="Aktueller Beruf der Person")

# LLM mit Schema binden + with_retry gegen transiente API-Fehler
structured_llm = llm.with_structured_output(Person).with_retry(stop_after_attempt=MAX_RETRIES)
person_prompt = load_prompt("https://github.com/ralf-42/Agenten/blob/main/05_prompt/m06_person_extraction_prompt.md", mode="T")

# Aufruf - LLM gibt ein Person-Objekt zurueck
ergebnis = (person_prompt | structured_llm).invoke(
    {"text": "Emma Mueller ist 34 Jahre alt und arbeitet als Aerztin."}
)

mprint("## Ergebnis")
mprint(f"**Typ:** `{type(ergebnis).__name__}` - kein String, sondern ein Python-Objekt")
mprint(f"**Name:** {ergebnis.name}")
mprint(f"**Alter:** {ergebnis.alter}")
mprint(f"**Beruf:** {ergebnis.beruf}")

# 4 | Praktische Anwendungen
---



Zwei häufige Anwendungsfälle:

**Informationsextraktion** – strukturierte Daten aus Freitext extrahieren

**Klassifikation** – Text in vordefinierte Kategorien einordnen (mit `Literal[...]`)

In [ ]:
class Ticket(BaseModel):
    """Klassifiziertes Support-Ticket."""
    kategorie: Literal["billing", "bug", "feature", "howto"] = Field(
        description="Kategorie: billing=Abrechnung, bug=Fehler, feature=Funktionswunsch, howto=Anleitung"
    )
    prioritaet: Literal["niedrig", "mittel", "hoch"] = Field(
        description="Prioritaet basierend auf Dringlichkeit und Auswirkung"
    )
    zusammenfassung: str = Field(
        description="Kurze Zusammenfassung des Anliegens in einem Satz"
    )

classifier = llm.with_structured_output(Ticket).with_retry(stop_after_attempt=MAX_RETRIES)
ticket_prompt = load_prompt("https://github.com/ralf-42/Agenten/blob/main/05_prompt/m06_ticket_classification_prompt.md", mode="T")

tickets = [
    "Meine Kreditkarte wurde zweimal belastet!",
    "Die App stuerzt ab, wenn ich auf Speichern klicke.",
    "Waere es moeglich, einen Dark Mode hinzuzufuegen?",
]

mprint("## Klassifizierte Tickets")
for text in tickets:
    t = (ticket_prompt | classifier).invoke({"ticket": text})
    mprint("")
    mprint(f"**Ticket:** {text}")
    mprint(f"→ `{t.kategorie}` | Prioritaet: `{t.prioritaet}`")
    mprint(f"→ {t.zusammenfassung}")

# 5 | Structured Output in der Praxis
---



Structured Output eignet sich überall dort, wo das LLM-Ergebnis direkt weiterverarbeitet werden soll:
- Formular-Ausfüllung aus Freitext
- Analyse-Pipelines mit definierten Ausgabeformaten
- Bewertung und Scoring von Texten

Der Unterschied zu freiem Text: Das Ergebnis ist **sofort als Python-Objekt nutzbar** – kein Parsing, keine Fehlerbehandlung für Format-Abweichungen.

In [ ]:
class Adresse(BaseModel):
    """Postadresse."""
    strasse: str = Field(description="Strasse und Hausnummer")
    ort: str = Field(description="Ort oder Stadt")
    plz: Optional[str] = Field(default=None, description="Postleitzahl, falls genannt")

class Kontakt(BaseModel):
    """Extrahierte Kontaktdaten aus einem Text."""
    name: str = Field(description="Vollstaendiger Name")
    email: Optional[str] = Field(default=None, description="E-Mail-Adresse, falls genannt")
    telefon: Optional[str] = Field(default=None, description="Telefonnummer, falls genannt")
    adresse: Optional[Adresse] = Field(default=None, description="Adresse, falls genannt")

kontakt_llm = llm.with_structured_output(Kontakt).with_retry(stop_after_attempt=MAX_RETRIES)
kontakt_prompt = load_prompt("https://github.com/ralf-42/Agenten/blob/main/05_prompt/m06_kontakt_extraction_prompt.md", mode="T")

text = """
Bitte kontaktieren Sie Dr. Anna Schmidt unter anna.schmidt@klinik.de
oder telefonisch unter 089 / 123 456 78.
Briefpost: Klinikstrasse 12, 80333 Muenchen.
"""

k = (kontakt_prompt | kontakt_llm).invoke({"text": text})

mprint("## Extrahierte Kontaktdaten")
mprint(f"**Name:** {k.name}")
mprint(f"**E-Mail:** {k.email}")
mprint(f"**Telefon:** {k.telefon}")
if k.adresse:
    mprint(f"**Adresse:** {k.adresse.strasse}, {k.adresse.plz} {k.adresse.ort}")

# 6 | LangSmith: Structured Output Traces
---



LangSmith macht Structured-Output-Aufrufe vollständig sichtbar:
- Das gesendete **JSON Schema** (Pydantic-Modell)
- Die **LLM-Antwort** vor der Validierung
- Das **valide Python-Objekt** nach der Validierung


In [ ]:
# 6.1 Structured Output Trace in LangSmith
class Produktbewertung(BaseModel):
    """Strukturierte Produktbewertung."""
    produkt: str = Field(description="Name des bewerteten Produkts")
    bewertung: int = Field(description="Bewertung von 1 (schlecht) bis 5 (sehr gut)")
    staerken: List[str] = Field(description="Positive Aspekte des Produkts (2-3 Punkte)")
    schwaechen: List[str] = Field(description="Negative Aspekte des Produkts (1-2 Punkte)")

# 1. Tracing-Konfiguration vorab festlegen
run_cfg = {
    "run_name": "M06_Kap6_StructuredTrace",
    "tags": ["M06", "structured-output", "langsmith"]
}

# 2. with_structured_output() + with_retry() + with_config()
bewertungs_llm = (
    llm
    .with_structured_output(Produktbewertung)
    .with_retry(stop_after_attempt=MAX_RETRIES)
    .with_config(**run_cfg)
)
bewertung_prompt = load_prompt("https://github.com/ralf-42/Agenten/blob/main/05_prompt/m06_produktbewertung_prompt.md", mode="T")

rezension = """
Das MacBook Pro M3 ist ein beeindruckendes Geraet mit atemberaubender Performance
und hervorragender Akkulaufzeit. Der Preis ist allerdings sehr hoch und die
Anschlussauswahl bleibt nach wie vor limitiert.
"""
ergebnis = (bewertung_prompt | bewertungs_llm).invoke({"rezension": rezension})

mprint("## Trace erstellt")
mprint(f"**Produkt:** {ergebnis.produkt}")
mprint(f"**Bewertung:** {'⭐' * ergebnis.bewertung} ({ergebnis.bewertung}/5)")
mprint(f"**Staerken:** {', '.join(ergebnis.staerken)}")
mprint(f"**Schwaechen:** {', '.join(ergebnis.schwaechen)}")

In [ ]:
#@title
#@markdown   <p><font size="4" color='green'>  LangSmith Trace-Analyse</font> </br></p>

import time as _t; _t.sleep(2)
show_trace("M06-Structured-Output", limit=3, show_steps=True)

# A | Aufgabe
---



Die Aufgabestellungen unten bieten Anregungen, Sie können aber auch gerne eine andere Herausforderung angehen.



<p><font color='black' size="5">
Strukturierte Informationsextraktion für Ihren Arbeitskontext
</font></p>

Definieren Sie ein Pydantic-Schema für einen Dokument- oder Texttyp aus Ihrem Arbeitsumfeld (z. B. E-Mail, Bericht, Protokoll, Anfrage) und extrahieren Sie strukturierte Daten daraus.

**Mindest-Anforderungen:**
- Mindestens 4 Felder mit `Field(description=...)`
- Mindestens 1 optionales Feld (`Optional[...]`)
- Mindestens 3 Test-Texte (unterschiedliche Fälle)

**Optionale Teilaufgaben:**
- Verwenden Sie `Literal[...]` für ein Klassifikationsfeld (z. B. Kategorie, Status)
- Bauen Sie ein Nested Schema mit einem Sub-Modell
- Kombinieren Sie Extraktion und Klassifikation in einem Schema
- Aktivieren Sie LangSmith-Tracing und inspizieren Sie das gesendete JSON Schema